## Converting irc Keras weights into PyTorch and compiling to Torchscript

Things to note: 
- the model was originally trained in Tensorflow (v1) so the first step is to convert it to a PyTorch compatible format
- the model was trained on a GPU so we need to load weights and re-compile to CPU
- it's important to check what version of torchvision (if used here) and torch you're running in this notebook environment & be sure they match the versions pinned in the deployment container's Dockerfile
- it's important to know which version of efficientnet was used in training (in irc's case it was [EfficientNetV2](http://pytorch.org/vision/main/models/efficientnetv2.html)) and the size of the inputs (224x224).

In [1]:
ORIGINAL_MODEL_PATH = "./model-weights/IRC_2.h5"

In [ ]:
# # hack to change model config from keras 2->3 compliant
# import h5py
# f = h5py.File(ORIGINAL_MODEL_PATH, mode="r+")
# model_config_string = f.attrs.get("model_config")
# print("Before fix, model_config contains 'groups':", model_config_string.find('"groups": 1,') != -1)
# if model_config_string.find('"groups": 1,') != -1:
#     model_config_string = model_config_string.replace('"groups": 1,', '')
#     f.attrs.modify('model_config', model_config_string)
#     f.flush()
#     model_config_string = f.attrs.get("model_config")
#     assert model_config_string.find('"groups": 1,') == -1

# print("After fix, model_config contains 'groups':", model_config_string.find('"groups": 1,') != -1)
# f.close()

In [8]:
import tensorflow as tf
# from tensorflow_addons.losses import SigmoidFocalCrossEntropy
import tensorflow_addons as tfa
import tf2onnx


# Convert Keras (.h5 to ONNX)
model = tf.keras.models.load_model(ORIGINAL_MODEL_PATH, custom_objects={'SigmoidFocalCrossEntropy': tfa.losses.SigmoidFocalCrossEntropy})
spec = (tf.TensorSpec((None, *model.input.shape[1:]), tf.float32, name="input"),)
output_path = "./model-weights/model.onnx"
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, output_path=output_path)

/opt/homebrew/Caskroom/miniforge/base/envs/irc-classifier/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/irc-classifier/lib/python3.10/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.9.1 and is not supported. 
Some things might work, some things might

In [10]:
# Convert ONNX to PyTorch
from onnx2pytorch import ConvertModel
import onnx

onnx_model = onnx.load(output_path)
pytorch_model = ConvertModel(onnx_model)

/opt/homebrew/Caskroom/miniforge/base/envs/irc-classifier/lib/python3.10/site-packages/onnx2pytorch/convert/layer.py:30: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at  /Users/runner/miniforge3/conda-bld/pytorch-recipe_1660136240338/work/torch/csrc/utils/tensor_numpy.cpp:178.)
  layer.weight.data = torch.from_numpy(numpy_helper.to_array(weight))


In [20]:
# Save out the whole model for future inference deployment
# https://pytorch.org/tutorials/beginner/saving_loading_models.html
import torch

compiled_path = './model-weights/irc_compiled_cpu.pt'

# model_scripted = torch.jit.script(pytorch_model) # Export to TorchScript
# model_scripted.save(compiled_path) # Save

example_input = torch.randn(1, 224, 224, 3)  # Replace input_shape appropriately
traced_model = torch.jit.trace(pytorch_model, example_input)

/opt/homebrew/Caskroom/miniforge/base/envs/irc-classifier/lib/python3.10/site-packages/onnx2pytorch/operations/squeeze.py:24: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  for dim in sorted(dims, reverse=True):
/opt/homebrew/Caskroom/miniforge/base/envs/irc-classifier/lib/python3.10/site-packages/onnx2pytorch/operations/squeeze.py:24: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might lead to errors or silently give incorrect results.
  for dim in sorted(dims, reverse=True):
/opt/homebrew/Caskroom/miniforge/base/envs/irc-classifier/lib/python3.10/site-packages/onnx2pytorch/operations/squeeze.py:24: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. W

In [ ]:
# Test inference

output = traced_model(example_input)
print(output)

tensor([[5.4554e-06, 2.0870e-06, 2.7595e-02, 5.5713e-03, 8.4770e-01, 1.5126e-02,
         2.1556e-04, 6.8486e-05, 5.1339e-02, 2.2230e-02, 1.6951e-02, 6.0459e-04,
         3.6771e-03, 4.6029e-03, 2.1190e-03, 2.1876e-03]],
       grad_fn=<SoftmaxBackward0>)


In [22]:
traced_model.save(compiled_path) # Save